# Análise Estatíticas Descritvas por Fator

## Importação das Bibliotecas

In [ ]:
import sys
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# 1. Configurações gerais
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# 2. Ativação do autoreload via API do IPython (compatível e sem SyntaxError no VS Code)
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

# 3. Adiciona a pasta scripts ao caminho do Python ANTES de importar os módulos locais
sys.path.append(str(Path.cwd().parent / 'scripts'))

# 4. Importação dos módulos do seu projeto
from prepara_dados_fator import prepara_dados_fator
from executa_teste_quiquadrado import executa_teste_quiquadrado
from verifica_pressupostos_anova import verifica_pressupostos_anova
from executa_anova_oneway import executa_anova_oneway
from executa_kruskal_wallis import executa_kruskal_wallis
from executa_tukey_hsd import executa_tukey_hsd
from executa_posthoc_mannwhitney import executa_posthoc_mannwhitney
from plota_boxplot_fator import plota_boxplot_fator
from plota_barras_ponderadas import plota_barras_ponderadas
from formata_tabela import formata_tabela


## Leitura do arquivo de dados

In [2]:
df = pd.read_parquet('../Data/Candidatos_Tratados.parquet')

## Análise Estatística Descritiva por Fator

As seguintes etapas serão seguidas para todos os fatores analisados:

1. Preparação da base de dados e geração da tabela estatística descritiva

Nesta etapa, é realizado o agrupamento da taxa de conclusão real de cada Instituto Federal (nível institucional). Os resultados são organizados em uma tabela comparando as medidas descritivas (mínimo, mediana, média simples e desvio-padrão dos IFs) com a Taxa Ponderada Global dos alunos na Rede Federal.

2. Teste Qui-Quadrado de Independência (Nível Discente)

Neste teste, cada estudante possui o mesmo peso na análise, independentemente de a sua instituição de origem ter 500 ou 50 mil alunos. O objetivo é responder se a probabilidade de conclusão ou evasão do aluno está estatisticamente associada ao fator sociodemográfico avaliado.

3. Teste de normalidade

Ao se aplicar a ANOVA deve-se verificar se pressupostos de Gauss-Markov (normalidade dos resíduos via Shapiro-Wilk e homogeneidade das variâncias via Levene) são atendidos. 

Neste estudo, constatou-se que apenas a variável Região satisfez ambos os pressupostos, autorizando o uso da ANOVA paramétrica de Fisher e do teste post-hoc de Tukey HSD. Para as variáveis Etnia, Renda, Faixa Etária e Sexo, em virtude da violação da normalidade, adotou-se o teste não-paramétrico de Kruskal-Wallis (ANOVA por postos), assegurando a robustez inferencial das conclusões.

4. Análise de Variância (ANOVA One-Way) ou Teste de Kruskal-Wallis (Nível Institucional)

A escolha entre o método paramétrico ou não-paramétrico depende da verificação formal dos pressupostos da ANOVA (normalidade dos resíduos via Shapiro-Wilk e homocedasticidade via Levene). Caso ambos os pressupostos sejam atendidos, aplica-se a ANOVA de Fisher; caso algum ou ambos sejam violados, aplica-se o Teste de Kruskal-Wallis (ANOVA por postos). Ambos os testes avaliam se a influência do fator sobre as taxas institucionais é estatisticamente significante. A diferença fundamental em relação ao Qui-Quadrado é a base de observação: enquanto no Qui-Quadrado a unidade é o estudante, na ANOVA e no Kruskal-Wallis a unidade é a taxa de cada Instituto Federal, tendo cada instituição o mesmo peso independente do número de estudantes matriculados.

4. Comparações Múltiplas Par a Par (Pós-teste)

Quando o teste global indicar diferença estatisticamente significante ($p < 0,05$), realiza-se o teste par a par para identificar quais categorias diferem entre si: via Tukey HSD (se o modelo for a ANOVA paramétrica) ou via Teste de Mann-Whitney com correção de Bonferroni (se o modelo for o Kruskal-Wallis não-paramétrico).

5. Visualização Gráfica (Boxplot com Strip Plot)

Elaboração de gráfico boxplot para visualização da dispersão das taxas das instituições, destacando as medianas, a média de cada grupo, cada IF individualizado e a linha de referência da média nacional.


    

In [3]:
fatores = ['REG', 'ETNIA', 'FAIXA_ETARIA', 'RENDA', 'SEXO']

for fator in fatores:

    print("=" * 70)
    print(f"ANÁLISE ESTATÍSTICA DESCRITIVA PARA O {fator}")
    print("=" * 70)
    print("\n")

    # Preparar a base e pressupostos
    df_inst, tab_fator = prepara_dados_fator(df, fator)
    p_shapiro, p_levene = verifica_pressupostos_anova(df_inst, fator)

    # Gera tabela
    formata_tabela(tab_fator, fator)
    print("\n")

    # Teste Qui-Quadrado de Independência (Nível Discente)
    chi2, p_chi2 = executa_teste_quiquadrado(df, fator)
    if p_chi2 < 0.05:
        print(f"A conclusão do aluno é DEPENDENTE de {fator} (p < 0.05).")
    else:
        print(f"Não há associação estatisticamente significante detectada (p >= 0.05).")
    print("\n")

    # Gráfico de Barras Horizontais com a Taxa Ponderada Global (Nível Discente)
    plota_barras_ponderadas(df_inst, tab_fator, fator, fator)


    if p_shapiro > 0.05 and p_levene > 0.05:
        # Análise de Variância (ANOVA One-Way)
        f_stat, p_anova = executa_anova_oneway(df_inst, fator)
        print(f"ANOVA One-Way: F = {f_stat:.4f} | p-valor = {p_anova:.4e}")
        print("\n")
        
        if p_anova < 0.05:
            df_tukey = executa_tukey_hsd(df_inst, fator)
            formata_tabela(df_tukey, 'Comparação')
        else:
            print(f"Não há diferença estatisticamente significante entre as médias de {fator} (p >= 0.05).")
    else:
        # Teste de Kruskal-Wallis (Nível Institucional)
        h_stat, p_kruskal = executa_kruskal_wallis(df_inst, fator)
        print(f"Kruskal-Wallis: H = {h_stat:.4f} | p-valor = {p_kruskal:.4e}")

        if p_kruskal < 0.05:
            df_posthoc = executa_posthoc_mannwhitney(df_inst, fator)
            formata_tabela(df_posthoc, 'Comparação')
        else:
            print(f"Não há diferença estatisticamente significante entre as medianas de {fator}.")

    print("\n")


    # Gráfico Boxplot da distribuição dos IFs por Região
    plota_boxplot_fator(
        df=df,
        df_inst=df_inst,
        coluna_fator= fator,
        titulo_fator= fator
    )
    print("\n")
        

ANÁLISE ESTATÍSTICA DESCRITIVA PARA O REG


REG             N_IFS  MEDIA  DESVIO_PADRAO  MEDIANA    MIN    MAX  TAXA_PONDERADA
----------------------------------------------------------------------------------
Sul                 6  59.98          17.37    53.95  41.74  81.57           78.52
Sudeste             9  59.79          14.52    53.69  43.96  80.42           63.44
Norte               7  48.88           5.38    45.73  44.18  57.36           48.95
Centro-Oeste        5  47.23          14.06    45.04  35.05  69.87           41.97
Nordeste           11  42.38           7.38    42.47  30.29  56.39           43.64


A conclusão do aluno é DEPENDENTE de REG (p < 0.05).




NameError: name 'plota_barras_ponderadas' is not defined